# Data Cleaning — Beginner Notebook

Notebook interaktif untuk kasus pemula: membersihkan data penjualan sederhana.
**Petunjuk:** File > Save a copy in Drive sebelum mulai. Jalankan sel satu per satu (Shift+Enter).

## 1) Muat data sample
Data: `06-data-cleaning/data/sample_sales.csv`

In [ ]:
import pandas as pd
df = pd.read_csv('06-data-cleaning/data/sample_sales.csv')
df.head()

## 2) Inspeksi singkat: tipe & missing values

In [ ]:
df.info()
print('
Missing per kolom:')
print(df.isna().sum())

## 3) Bersihkan string & parsing tipe
- parse tanggal dengan `pd.to_datetime(..., errors='coerce')`
- ubah kolom numerik dengan `pd.to_numeric(..., errors='coerce')`

In [ ]:
# Trim strings
str_cols = df.select_dtypes(include=['object']).columns
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

# Parse types
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df.head()

## 4) Tangani missing values (strategi contoh)

In [ ]:
# Drop baris tanpa order_id atau order_date
df = df.dropna(subset=['order_id','order_date'])

# Fill quantity kosong dengan 1
df['quantity'] = df['quantity'].fillna(1).astype(int)

# Fill price kosong dengan median per product
df['price'] = df['price'].fillna(df.groupby('product')['price'].transform('median'))

df.isna().sum()

## 5) Hapus duplikat & buat kolom terhitung

In [ ]:
df = df.drop_duplicates(subset=['order_id'], keep='first')
df['total'] = df['quantity'] * df['price']
df.head()

## 6) Validasi cepat & simpan hasil

In [ ]:
# Validasi sederhana
assert (df['quantity'] >= 0).all(), 'Quantity negatif ditemukan'
assert (df['price'] >= 0).all(), 'Price negatif ditemukan'

# Simpan
df.to_csv('06-data-cleaning/data/sample_sales_cleaned.csv', index=False)
print('Selesai. File disimpan: 06-data-cleaning/data/sample_sales_cleaned.csv')

## Tugas latihan (opsional)
1) Isi sendiri strategi pengisian harga untuk produk yang tidak punya median (mis. gunakan global median).
2) Buat analisis ringkas: total per country dan 5 produk teratas menurut total penjualan.